# NB2 — Attention Visualization

In NB1 you pulled attention patterns out as `[head, dest, src]` tensors and stared at raw numbers.
That doesn't scale — GPT-2 small has **144 heads**. This notebook is about *seeing* them.

We'll use **`circuitsvis`**, the standard interactive-visualization companion to TransformerLens, to
render attention patterns, and we'll learn to recognize a few **canonical head shapes** by eye:

- **current-token heads** — attend to *themselves* (bright main diagonal),
- **previous-token heads** — attend to the token one position *back* (bright sub-diagonal),
- **duplicate-token heads** — attend to an *earlier copy* of the current token (we'll only foreshadow
  these here; they need repeated text, which is NB3's whole setup).

By the end you'll be able to point at a head and say what it's doing — the skill the entire induction
hunt rests on.

## 0. Setup

Same model setup as before, plus `circuitsvis` (imported as `cv`). If the visualizations don't render
in your environment, make sure the notebook is trusted / running in a real Jupyter frontend.

In [1]:
import torch
import circuitsvis as cv
from transformer_lens import HookedTransformer

torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2", device=device)
print("circuitsvis", cv.__version__, "| model on", device)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer
circuitsvis 1.41.0 | model on cuda


## 1. Cache a sentence

We need two things to visualize a layer: the **string tokens** (for axis labels) and the layer's
**attention pattern** `[head, dest, src]`. We use a sentence with a repeated name (`John`, `Mary`)
so later we can look for heads that connect the repeats.

In [2]:
text = "When Mary and John went to the store, John gave a bottle of milk to Mary because she was thirsty."
tokens = model.to_tokens(text)
str_tokens = model.to_str_tokens(text)   # list of strings, one per position (axis labels)

logits, cache = model.run_with_cache(tokens)
print("seq length:", len(str_tokens))
print(str_tokens)

seq length: 23
['<|endoftext|>', 'When', ' Mary', ' and', ' John', ' went', ' to', ' the', ' store', ',', ' John', ' gave', ' a', ' bottle', ' of', ' milk', ' to', ' Mary', ' because', ' she', ' was', ' thirsty', '.']


## 2. `circuitsvis.attention.attention_patterns`

The core call is:

```python
cv.attention.attention_patterns(tokens=str_tokens, attention=pattern)
```

where `pattern` is `[n_heads, dest, src]` — i.e. `cache["pattern", layer][0]` (we drop the batch
dim). It renders **all heads in the layer** with a head selector.

**How to read it:** it's the `[dest, src]` matrix from NB1, drawn as a grid. Each **row is a
destination** (query) token; the bright cells in that row are the **source** (key) tokens it attends
to. Because attention is causal, everything is on or below the diagonal.

Let's start with **layer 0**. Look at **head 1** — you should see an almost pure **main diagonal**:
every token attends to *itself*. That's a **current-token head**.

In [3]:
# All 12 heads of layer 0. Use the head selector; hover cells to read weights.
cv.attention.attention_patterns(tokens=str_tokens, attention=cache["pattern", 0][0])

## 3. A previous-token head

Now **layer 4**. Look at **head 11**: instead of the main diagonal, the bright line sits **one step
below** it — every token attends to the token *immediately before* it. That's a **previous-token
head**, and L4H11 is a famously clean one in GPT-2 small.

Keep this head in mind: previous-token heads are the **first half** of the induction circuit. In NB4
we'll see how a head like this feeds an induction head in a later layer.

In [4]:
# All 12 heads of layer 4 — select head 11 to see the sub-diagonal previous-token pattern.
cv.attention.attention_patterns(tokens=str_tokens, attention=cache["pattern", 4][0])

## 4. From eyeballs to numbers: the diagonal trick

Reading grids works but doesn't scale to 144 heads. We can turn each canonical shape into a **single
score** using tensor diagonals:

- **current-token score** = mean of the **main diagonal** of `pattern` (`offset=0`): how much each
  token attends to itself, averaged over positions.
- **previous-token score** = mean of the **first sub-diagonal** (`offset=-1`): how much each token
  attends to the one before it.

`tensor.diagonal(offset=k, dim1=-2, dim2=-1)` pulls out diagonal `k` of the last two dims. Here's the
current-token score for every head, with the top few printed — L0H1 should dominate.

### The key line, step by step

The whole reduction lives in one line:

```python
scores[L] = p.diagonal(dim1=-2, dim2=-1).mean(-1).cpu()
```

It walks a full attention pattern down to one number per head. Follow the shapes (say `seq = 23`,
`n_heads = 12`):

| expression | shape | what it holds |
|------------|-------|---------------|
| `p` | `[n_heads, dest, src]` = `[12, 23, 23]` | the layer's full attention pattern |
| `p.diagonal(dim1=-2, dim2=-1)` | `[12, 23]` | for each head, the values `p[h, i, i]` — each token's attention to **itself**, one per position |
| `.mean(-1)` | `[12]` | average over positions → **one current-token score per head** |
| `.cpu()` | `[12]` | move that tiny result off the GPU into the CPU `scores` tensor |

Piece by piece:

- **`dim1=-2, dim2=-1`** tells `diagonal` *which* two axes form the square matrix to walk down — the
  last two (`dest`, `src`). The `head` axis is left untouched, so you get a diagonal *per head*, not
  one diagonal over the whole tensor.
- **default `offset=0`** = the main diagonal, where `dest == src` (attend-to-self). Switch it to
  `offset=-1` and you read the `dest = src + 1` line instead (attend to the previous token) — that
  single change is the entire difference between this and the practice function.
- **`.mean(-1)`** collapses the 23 per-position values into one number. Because each row of `p` sums
  to 1, every diagonal value is a probability in `[0, 1]`, so the mean is too: `0.91` reads as
  "attends to itself ~91% of the time, averaged over the sentence."
- **`.cpu()`** moves the small `[12]` result to the host so it can be assigned into `scores` (created
  with `torch.zeros`, so CPU-resident). The heavy work stayed on the GPU; only the summary crosses
  over. See the device discussion from earlier.

One subtlety: with `offset=-1` the diagonal has length `seq - 1 = 22`, not 23 (there's no token
before position 0). `.mean(-1)` handles that transparently — it just averages 22 values instead of 23.

In [5]:
def current_token_score(cache, model):
    """Mean main-diagonal attention for every head -> [n_layers, n_heads]."""
    nL, nH = model.cfg.n_layers, model.cfg.n_heads
    scores = torch.zeros(nL, nH)                        # CPU tensor, one score per (layer, head)
    for L in range(nL):
        p = cache["pattern", L][0]                      # [head, dest, src], e.g. [12, 23, 23]
        # Walk the reduction right-to-left:
        #   p.diagonal(dim1=-2, dim2=-1) -> [head, 23]  each token's attention to ITSELF (p[h,i,i])
        #   .mean(-1)                    -> [head]       average over the 23 positions: one score/head
        #   .cpu()                       -> [head]       move the small result to CPU to store below
        scores[L] = p.diagonal(dim1=-2, dim2=-1).mean(-1).cpu()
    return scores

cur = current_token_score(cache, model)

# Print the top-5 heads as (layer, head, score).
flat = cur.flatten()
for i in flat.topk(5).indices:
    L, H = int(i) // model.cfg.n_heads, int(i) % model.cfg.n_heads
    print(f"L{L}H{H}: current-token score = {flat[i]:.3f}")

L0H1: current-token score = 0.907
L0H3: current-token score = 0.839
L0H5: current-token score = 0.714
L1H11: current-token score = 0.674
L4H7: current-token score = 0.554


## Canonical head shapes — reference

| Shape | Where the bright cells are | `diagonal` offset | Example (GPT-2 small) |
|-------|---------------------------|-------------------|-----------------------|
| current-token | main diagonal (attend to self) | `0` | **L0H1** |
| previous-token | one below diagonal (attend to *i−1*) | `−1` | **L4H11** |
| duplicate-token | at an *earlier copy* of the current token | (needs repeats) | seen in NB3 |
| **induction** | at the token *after* the previous copy | (needs repeats) | the goal — NB3 |

Duplicate-token and induction heads only reveal themselves when the input **repeats**, because they
act on earlier occurrences of the current token. That's exactly why NB3 feeds the model
repeated-random-token sequences.

## 5. A third detector: first-token (attention-sink) heads

One more shape worth scoring, because it's the biggest source of *false positives* when hunting for
interesting heads. Many heads, when they have nothing relevant to attend to, dump their attention on
**position 0 — the BOS token** (`<|endoftext|>`). Softmax forces every row to sum to 1, so a head
with no active trigger has to put its weight *somewhere*; the model uses BOS as a **no-op / rest
slot**. These are **attention-sink** heads, and their signature is a bright **first column** (`src=0`).

The detector is the same trick with a twist: instead of a diagonal, average **column 0**:

```
first-token score = mean over dest of pattern[head, dest, 0]
```

Let's score every head on our plain sentence — and notice *which* heads top the list.

In [6]:
def first_token_score(cache, model):
    """Mean attention to position 0 (the BOS token) for every head -> [n_layers, n_heads]."""
    nL, nH = model.cfg.n_layers, model.cfg.n_heads
    scores = torch.zeros(nL, nH)
    for L in range(nL):
        p = cache["pattern", L][0]              # [head, dest, src]
        scores[L] = p[:, :, 0].mean(-1).cpu()   # column 0 = attention to BOS, averaged over dest
    return scores

first = first_token_score(cache, model)
flat = first.flatten()
for i in flat.topk(5).indices:
    L, H = int(i) // model.cfg.n_heads, int(i) % model.cfg.n_heads
    print(f"L{L}H{H}: first-token score = {flat[i]:.3f}")

L5H1: first-token score = 0.995
L7H2: first-token score = 0.993
L7H10: first-token score = 0.963
L8H1: first-token score = 0.962
L6H9: first-token score = 0.959


**Look closely at that list.** The top BOS-attenders on plain text — L5H1, L7H2, L7H10, L6H9 — are
*the very heads we'll later identify as induction heads* (NB3). That's not a coincidence: on ordinary
text there are no repeated tokens for them to induct on, so they sit in their **rest state, parked on
BOS**. Their interesting behaviour is invisible here.

This is exactly why NB3 abandons plain sentences and feeds the model **repeated-random-token
sequences**: only when the induction trigger (a duplicated token) is actually present do these heads
leave the BOS sink and reveal what they really do. Two lessons to carry forward:

1. **When scoring heads, watch out for the BOS sink** — it inflates scores and hides real behaviour.
   Serious analyses prepend BOS and then *ignore* attention to position 0.
2. **A head's function can be input-dependent.** A head that looks like it "just attends to BOS" may
   only be doing that for the input you happened to give it. Pick the input that triggers the
   behaviour you're hunting.

## Practice

Mirror the `current_token_score` function to build a **`previous_token_score`** and use it to find the
cleanest previous-token head in GPT-2 small.

Task:
1. Write `previous_token_score(cache, model)` returning a `[n_layers, n_heads]` tensor — the same as
   `current_token_score` but using the **first sub-diagonal** (`offset=-1`) instead of the main one.
2. Print the **top-5** heads by that score.
3. Confirm **L4H11** comes out on top, then open the layer-4 visualization above and verify head 11
   really is the sub-diagonal one.

Hint: the only change from `current_token_score` is `p.diagonal(offset=-1, dim1=-2, dim2=-1)`.

In [7]:
def previous_token_score(cache, model):
    """Mean main-diagonal attention for every head -> [n_layers, n_heads]."""
    nL, nH = model.cfg.n_layers, model.cfg.n_heads
    scores = torch.zeros(nL, nH)
    for L in range(nL):
        p = cache["pattern", L][0]                      # [head, dest, src]
        scores[L] = p.diagonal(offset=-1, dim1=-2, dim2=-1).mean(-1).cpu()   # mean over positions
    return scores

prev = previous_token_score(cache, model)

# Print the top-5 heads as (layer, head, score).
flat = prev.flatten()
for i in flat.topk(5).indices:
    L, H = int(i) // model.cfg.n_heads, int(i) % model.cfg.n_heads
    print(f"L{L}H{H}: previous-token score = {flat[i]:.3f}")
    


L4H11: previous-token score = 1.000
L2H2: previous-token score = 0.564
L3H2: previous-token score = 0.405
L3H7: previous-token score = 0.393
L2H9: previous-token score = 0.369


<details>
<summary>Reference solution (open after you've tried)</summary>

```python
def previous_token_score(cache, model):
    nL, nH = model.cfg.n_layers, model.cfg.n_heads
    scores = torch.zeros(nL, nH)
    for L in range(nL):
        p = cache["pattern", L][0]                                    # [head, dest, src]
        scores[L] = p.diagonal(offset=-1, dim1=-2, dim2=-1).mean(-1).cpu()
    return scores

prev = previous_token_score(cache, model)
flat = prev.flatten()
for i in flat.topk(5).indices:
    L, H = int(i) // model.cfg.n_heads, int(i) % model.cfg.n_heads
    print(f"L{L}H{H}: previous-token score = {flat[i]:.3f}")
```

Expected top: `L4H11` with a score near 1.0.

</details>

---
**Done?** Once your `previous_token_score` puts L4H11 on top and you've matched it to the picture,
ask me to review. Next is **NB3 — induction detection**, where we build repeated-random-token
sequences, watch the loss drop on the second copy, and write an **induction score** that finally
pinpoints the induction heads among all 144.